# Notebook 03 — Feature Engineering

**Purpose:** Take the cleaned Brooklyn dataset and enrich it with spatial features (subway accessibility, POI density), engineered PLUTO features (`building_age`, `far`, `unit_density`, commercial ratios), and a clean binary encoding of the FEMA flood-zone flag.

**Inputs:**
- `data/processed/brooklyn_clean.parquet` (265,507 rows × 27 cols, from Notebook 02)
- `data/raw/mta_subway_stations.csv`
- `data/raw/osm_pois.geojson` (8,980 features, 6 categories)

**Outputs:**
- `data/processed/brooklyn_features.parquet`

**Downstream:** `04_eda.ipynb`, `05_clustering.ipynb`, `06_regression.ipynb`

**Author:** Delfin Aksu

## 1. Setup

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from sklearn.neighbors import BallTree

warnings.filterwarnings('ignore', category=FutureWarning)

In [2]:
PROJECT_ROOT = Path('..').resolve()
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'

CRS_WGS84 = 4326
CRS_NYSP  = 2263

OSM_CATEGORIES = ['cafe', 'restaurant', 'school', 'park', 'supermarket', 'convenience']

RADIUS_500M_FT  = 1640
RADIUS_1000M_FT = 3281

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 200)

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'500m radius in feet:  {RADIUS_500M_FT:>5}')
print(f'1000m radius in feet: {RADIUS_1000M_FT:>5}')

PROJECT_ROOT: C:\Users\Delfin AKSU\Desktop\urban-building-ml
500m radius in feet:   1640
1000m radius in feet:  3281


## 2. Load the Three Source Datasets

### 2.1 Cleaned Brooklyn PLUTO

In [3]:
clean_path = PROCESSED_DATA_DIR / 'brooklyn_clean.parquet'
buildings = pd.read_parquet(clean_path)
print(f'Loaded brooklyn_clean.parquet: {buildings.shape[0]:,} rows x {buildings.shape[1]} columns')

Loaded brooklyn_clean.parquet: 265,507 rows x 27 columns


In [4]:
buildings.head(3)

,bbl,borough,block,lot,address,zipcode,cd,latitude,longitude,bldgclass,landuse,lotarea,bldgarea,comarea,resarea,numbldgs,numfloors,unitsres,unitstotal,yearbuilt,yearalter1,assessland,assesstot,exempttot,builtfar,firm07_flag,is_extreme_value
0,3.007480e+09,BK,748,12,520 45 STREET,11220.0,307.0,40.647409,-74.007045,C0,2.0,2003.0,3060.0,0.0,3060.0,1.0,3.0,3.0,3.0,1920.0,0.0,31620.0,99180.0,0.0,1.53,NaN,False
1,3.006400e+09,BK,640,42,688 5 AVENUE,11215.0,307.0,40.661611,-73.993476,S2,4.0,1408.0,2520.0,832.0,1688.0,1.0,3.0,2.0,3.0,1932.0,0.0,11460.0,141360.0,0.0,1.79,NaN,False
2,3.058840e+09,BK,5884,48,97A 71 STREET,11209.0,310.0,40.635819,-74.032697,A5,1.0,1169.0,1142.0,0.0,1142.0,1.0,2.0,1.0,1.0,1935.0,0.0,8580.0,56040.0,1420.0,0.98,NaN,False


In [5]:
buildings.dtypes.to_frame('dtype')

,dtype
bbl,float64
borough,str
block,int64
lot,int64
address,str
zipcode,float64
cd,float64
latitude,float64
longitude,float64
bldgclass,str


### 2.2 MTA Subway Stations

In [6]:
subway = pd.read_csv(RAW_DATA_DIR / 'mta_subway_stations.csv')
print(f'Loaded mta_subway_stations.csv: {subway.shape[0]} stations x {subway.shape[1]} columns')

Loaded mta_subway_stations.csv: 496 stations x 19 columns


In [7]:
subway[['GTFS Stop ID', 'Station ID', 'Stop Name', 'Borough', 'GTFS Latitude', 'GTFS Longitude']].head(5)

,GTFS Stop ID,Station ID,Stop Name,Borough,GTFS Latitude,GTFS Longitude
0,R01,1,Astoria-Ditmars Blvd,Q,40.775036,-73.912034
1,R03,2,Astoria Blvd,Q,40.770258,-73.917843
2,R04,3,30 Av,Q,40.766779,-73.921479
3,R05,4,Broadway,Q,40.761820,-73.925508
4,R06,5,36 Av,Q,40.756804,-73.929575


**Note:** All 5 boroughs included — Brooklyn-edge buildings may have a Queens/Manhattan station as nearest.

### 2.3 OpenStreetMap POIs

In [8]:
with open(RAW_DATA_DIR / 'osm_pois.geojson', 'r', encoding='utf-8') as f:
    osm_geo = json.load(f)

pois = pd.DataFrame([
    {'osm_id': feat['properties']['osm_id'], 'category': feat['properties']['category'],
     'name': feat['properties'].get('name'),
     'lon': feat['geometry']['coordinates'][0], 'lat': feat['geometry']['coordinates'][1]}
    for feat in osm_geo['features']
])

print(f'Loaded osm_pois.geojson: {len(pois):,} POIs across {pois["category"].nunique()} categories')

Loaded osm_pois.geojson: 8,980 POIs across 6 categories


In [9]:
pois['category'].value_counts().reindex(OSM_CATEGORIES, fill_value=0).to_frame('count')

,count
category,
cafe,1439
restaurant,3858
school,788
park,780
supermarket,541
convenience,1574


### 2.4 Load summary

In [10]:
print('Source            | Records   | Columns | Geometry')
print('-' * 65)
print(f'buildings (PLUTO) | {len(buildings):>7,} | {buildings.shape[1]:>7} | latitude/longitude (EPSG:4326)')
print(f'subway (MTA)      | {len(subway):>7,} | {subway.shape[1]:>7} | GTFS Latitude/Longitude (EPSG:4326)')
print(f'pois (OSM)        | {len(pois):>7,} | {pois.shape[1]:>7} | lon/lat (EPSG:4326)')

Source            | Records   | Columns | Geometry
-----------------------------------------------------------------
buildings (PLUTO) | 265,507 |      27 | latitude/longitude (EPSG:4326)
subway (MTA)      |     496 |      19 | GTFS Latitude/Longitude (EPSG:4326)
pois (OSM)        |   8,980 |       5 | lon/lat (EPSG:4326)


## 3. Build GeoDataFrames and Reproject to NY State Plane

### 3.1 Build GeoDataFrames in EPSG:4326

In [11]:
buildings_gdf = gpd.GeoDataFrame(buildings, geometry=gpd.points_from_xy(buildings['longitude'], buildings['latitude']), crs=CRS_WGS84)
subway_gdf    = gpd.GeoDataFrame(subway,    geometry=gpd.points_from_xy(subway['GTFS Longitude'], subway['GTFS Latitude']), crs=CRS_WGS84)
pois_gdf      = gpd.GeoDataFrame(pois,      geometry=gpd.points_from_xy(pois['lon'], pois['lat']),   crs=CRS_WGS84)

print('GeoDataFrames built in EPSG:4326:')
print(f'  buildings_gdf: {len(buildings_gdf):>7,} features, crs={buildings_gdf.crs}')
print(f'  subway_gdf:    {len(subway_gdf):>7,} features, crs={subway_gdf.crs}')
print(f'  pois_gdf:      {len(pois_gdf):>7,} features, crs={pois_gdf.crs}')

GeoDataFrames built in EPSG:4326:
  buildings_gdf: 265,507 features, crs=EPSG:4326
  subway_gdf:        496 features, crs=EPSG:4326
  pois_gdf:        8,980 features, crs=EPSG:4326


### 3.2 Quick geometry sanity check

In [12]:
for name, g in [('buildings', buildings_gdf), ('subway', subway_gdf), ('pois', pois_gdf)]:
    sample = g.geometry.iloc[0]
    print(f'  {name:9s} first geometry: {sample} (type: {type(sample).__name__})')

  buildings first geometry: POINT (-74.0070451 40.647409) (type: Point)
  subway    first geometry: POINT (-73.912034 40.775036) (type: Point)
  pois      first geometry: POINT (-74.0000236 40.5952532) (type: Point)


### 3.3 Reproject to EPSG:2263

In [13]:
buildings_gdf = buildings_gdf.to_crs(CRS_NYSP)
subway_gdf    = subway_gdf.to_crs(CRS_NYSP)
pois_gdf      = pois_gdf.to_crs(CRS_NYSP)

print('Reprojected to EPSG:2263:')
print(f'  buildings_gdf: crs={buildings_gdf.crs.to_string()}')
print(f'  subway_gdf:    crs={subway_gdf.crs.to_string()}')
print(f'  pois_gdf:      crs={pois_gdf.crs.to_string()}')

Reprojected to EPSG:2263:
  buildings_gdf: crs=EPSG:2263
  subway_gdf:    crs=EPSG:2263
  pois_gdf:      crs=EPSG:2263


### 3.4 Verify the reprojection

In [14]:
print(f'Sample building geometry (EPSG:2263): x={buildings_gdf.geometry.iloc[0].x:,.0f}  y={buildings_gdf.geometry.iloc[0].y:,.0f}')
print(f'Sample subway geometry   (EPSG:2263): x={subway_gdf.geometry.iloc[0].x:,.0f}  y={subway_gdf.geometry.iloc[0].y:,.0f}')
print(f'Sample POI geometry      (EPSG:2263): x={pois_gdf.geometry.iloc[0].x:,.0f}  y={pois_gdf.geometry.iloc[0].y:,.0f}')

Sample building geometry (EPSG:2263): x=982,295  y=175,145
Sample subway geometry   (EPSG:2263): x=1,008,614  y=221,656
Sample POI geometry      (EPSG:2263): x=984,243  y=156,143


## 4. Build Spatial Indexes (`BallTree`)

### 4.1 Subway BallTree

In [15]:
subway_coords = np.column_stack([subway_gdf.geometry.x, subway_gdf.geometry.y])
print(f'subway_coords shape: {subway_coords.shape}')
subway_tree = BallTree(subway_coords, metric='euclidean')
print(f'subway_tree built: {subway_tree}')

subway_coords shape: (496, 2)
subway_tree built: <sklearn.neighbors._ball_tree.BallTree object at 0x0000016C845ACC80>


### 4.2 One BallTree per POI category

In [16]:
poi_trees = {}
for cat in OSM_CATEGORIES:
    cat_gdf = pois_gdf[pois_gdf['category'] == cat]
    n = len(cat_gdf)
    if n == 0:
        poi_trees[cat] = None
        print(f'  {cat:11s}: 0 POIs -- skipping')
        continue
    coords = np.column_stack([cat_gdf.geometry.x, cat_gdf.geometry.y])
    poi_trees[cat] = BallTree(coords, metric='euclidean')
    print(f'  {cat:11s}: {n:>5,} POIs -- BallTree built')

print()
print(f'Total trees: {sum(1 for t in poi_trees.values() if t is not None)} (of {len(OSM_CATEGORIES)} categories)')

  cafe       : 1,439 POIs -- BallTree built
  restaurant : 3,858 POIs -- BallTree built
  school     :   788 POIs -- BallTree built
  park       :   780 POIs -- BallTree built
  supermarket:   541 POIs -- BallTree built
  convenience: 1,574 POIs -- BallTree built

Total trees: 6 (of 6 categories)


### 4.3 Verify a query works end-to-end

In [17]:
test_pt = np.array([[buildings_gdf.geometry.iloc[0].x, buildings_gdf.geometry.iloc[0].y]])
nearest_dist, nearest_idx = subway_tree.query(test_pt, k=1)
nearest_station = subway_gdf.iloc[int(nearest_idx[0,0])]
print(f'Nearest subway station: {nearest_station["Stop Name"]} ({nearest_station["Borough"]})')
print(f'Distance (feet):  {nearest_dist[0,0]:,.1f}')
print(f'Distance (metres): {nearest_dist[0,0] / 3.2808:,.1f}')
rest_within = poi_trees['restaurant'].query_radius(test_pt, r=RADIUS_500M_FT, count_only=True)
print(f'Restaurants within 500m: {int(rest_within[0])}')

Nearest subway station: 45 St (Bk)
Distance (feet):  992.9
Distance (metres): 302.6
Restaurants within 500m: 23


## 5. Subway Accessibility Features

### 5.1 Compute the three features

In [18]:
building_coords = np.column_stack([buildings_gdf.geometry.x, buildings_gdf.geometry.y])
print(f'building_coords shape: {building_coords.shape}')

nearest_dists, _ = subway_tree.query(building_coords, k=1)
buildings_gdf['nearest_subway_dist'] = nearest_dists.flatten()
buildings_gdf['subway_count_500m']  = subway_tree.query_radius(building_coords, r=RADIUS_500M_FT,  count_only=True)
buildings_gdf['subway_count_1000m'] = subway_tree.query_radius(building_coords, r=RADIUS_1000M_FT, count_only=True)

print(f'Computed 3 subway features for {len(buildings_gdf):,} buildings.')

building_coords shape: (265507, 2)
Computed 3 subway features for 265,507 buildings.


### 5.2 Distribution check

In [19]:
subway_feats = ['nearest_subway_dist', 'subway_count_500m', 'subway_count_1000m']
buildings_gdf[subway_feats].describe()

,nearest_subway_dist,subway_count_500m,subway_count_1000m
count,265507.000000,265507.000000,265507.000000
mean,2265.371094,0.860829,3.318801
std,2207.815723,0.985374,2.614200
min,9.760745,0.000000,0.000000
25%,928.459896,0.000000,1.000000
50%,1477.783057,1.000000,3.000000
75%,2615.363427,1.000000,5.000000
max,15920.240944,8.000000,17.000000


### 5.3 Sanity checks

In [20]:
checks = {
    'no NaN in nearest_subway_dist':           bool(buildings_gdf['nearest_subway_dist'].notna().all()),
    'no NaN in subway_count_500m':             bool(buildings_gdf['subway_count_500m'].notna().all()),
    'no NaN in subway_count_1000m':            bool(buildings_gdf['subway_count_1000m'].notna().all()),
    'all nearest_subway_dist > 0':             bool((buildings_gdf['nearest_subway_dist'] > 0).all()),
    'all nearest_subway_dist < 30000 ft':      bool((buildings_gdf['nearest_subway_dist'] < 30000).all()),
    'subway_count_1000m >= subway_count_500m': bool((buildings_gdf['subway_count_1000m'] >= buildings_gdf['subway_count_500m']).all()),
}
for key, ok in checks.items():
    print(f'  {key:42s}: {ok}')

  no NaN in nearest_subway_dist             : True
  no NaN in subway_count_500m               : True
  no NaN in subway_count_1000m              : True
  all nearest_subway_dist > 0               : True
  all nearest_subway_dist < 30000 ft        : True
  subway_count_1000m >= subway_count_500m   : True


### 5.4 Most / least transit-served buildings

In [21]:
best_idx = buildings_gdf.sort_values(['subway_count_1000m', 'nearest_subway_dist'], ascending=[False, True]).index[0]
worst_idx = buildings_gdf.sort_values('nearest_subway_dist', ascending=False).index[0]
show_cols = ['bbl', 'address', 'cd', 'nearest_subway_dist', 'subway_count_500m', 'subway_count_1000m']
print('Most transit-rich building:')
print(buildings_gdf.loc[best_idx, show_cols].to_string())
print()
print('Least transit-served building:')
print(buildings_gdf.loc[worst_idx, show_cols].to_string())

Most transit-rich building:
bbl                       3001960013.0
address                204 DEAN STREET
cd                               302.0
nearest_subway_dist         1172.35878
subway_count_500m                    2
subway_count_1000m                  17

Least transit-served building:
bbl                            3085910075.0
address                5031 FLATBUSH AVENUE
cd                                    356.0
nearest_subway_dist            15920.240944
subway_count_500m                         0
subway_count_1000m                        0


## 6. POI Density Features (6 categories × 500m radius)

**Why 500m?** Urban walkability literature consistently uses ~400–500m as the canonical "walking distance" (a 5-minute walk).

**Why per-category?** Different POI types convey different aspects of an urban location.

### 6.1 Compute one feature column per category

In [22]:
for cat in OSM_CATEGORIES:
    feat_name = f'{cat}_count_500m'
    if poi_trees[cat] is None:
        buildings_gdf[feat_name] = 0
        print(f'  {feat_name:25s}: filled with 0 (no POIs in category)')
        continue
    counts = poi_trees[cat].query_radius(building_coords, r=RADIUS_500M_FT, count_only=True)
    buildings_gdf[feat_name] = counts
    print(f'  {feat_name:25s}: computed (min={counts.min()}, max={counts.max()}, median={np.median(counts):.0f})')

  cafe_count_500m          : computed (min=0, max=43, median=1)
  restaurant_count_500m    : computed (min=0, max=105, median=4)
  school_count_500m        : computed (min=0, max=21, median=2)
  park_count_500m          : computed (min=0, max=20, median=1)
  supermarket_count_500m   : computed (min=0, max=11, median=1)
  convenience_count_500m   : computed (min=0, max=44, median=3)


### 6.2 Distribution check

In [23]:
poi_feat_cols = [f'{cat}_count_500m' for cat in OSM_CATEGORIES]
buildings_gdf[poi_feat_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
cafe_count_500m,265507.0,4.234269,6.466520,0.0,0.0,1.0,6.0,43.0
restaurant_count_500m,265507.0,9.892692,13.646946,0.0,1.0,4.0,14.0,105.0
school_count_500m,265507.0,3.016647,2.557546,0.0,1.0,2.0,4.0,21.0
park_count_500m,265507.0,1.941203,2.024935,0.0,1.0,1.0,3.0,20.0
supermarket_count_500m,265507.0,1.983119,1.891988,0.0,0.0,1.0,3.0,11.0
convenience_count_500m,265507.0,6.336737,7.374583,0.0,0.0,3.0,10.0,44.0


### 6.3 Sanity checks

In [24]:
checks = {f'{c}_count_500m: no NaN':       bool(buildings_gdf[f'{c}_count_500m'].notna().all()) for c in OSM_CATEGORIES}
checks.update({f'{c}_count_500m: all >= 0': bool((buildings_gdf[f'{c}_count_500m'] >= 0).all()) for c in OSM_CATEGORIES})
for key, ok in checks.items():
    print(f'  {key:36s}: {ok}')

  cafe_count_500m: no NaN             : True
  restaurant_count_500m: no NaN       : True
  school_count_500m: no NaN           : True
  park_count_500m: no NaN             : True
  supermarket_count_500m: no NaN      : True
  convenience_count_500m: no NaN      : True
  cafe_count_500m: all >= 0           : True
  restaurant_count_500m: all >= 0     : True
  school_count_500m: all >= 0         : True
  park_count_500m: all >= 0           : True
  supermarket_count_500m: all >= 0    : True
  convenience_count_500m: all >= 0    : True


### 6.4 Total POI exposure -- a derived view

In [25]:
buildings_gdf['poi_count_500m_total'] = buildings_gdf[poi_feat_cols].sum(axis=1)
buildings_gdf['poi_count_500m_total'].describe()

count    265507.000000
mean         27.404667
std          27.881130
min           0.000000
25%           7.000000
50%          16.000000
75%          39.000000
max         186.000000
Name: poi_count_500m_total, dtype: float64

### 6.5 Most POI-dense building

In [26]:
top_idx = buildings_gdf['poi_count_500m_total'].idxmax()
row = buildings_gdf.loc[top_idx, ['bbl', 'address', 'cd'] + poi_feat_cols + ['poi_count_500m_total']]
print('Building with the highest total POI density (500m disc):')
print(row.to_string())

Building with the highest total POI density (500m disc):
bbl                                  3023590020.0
address                   193 METROPOLITAN AVENUE
cd                                          301.0
cafe_count_500m                                43
restaurant_count_500m                         103
school_count_500m                              10
park_count_500m                                 7
supermarket_count_500m                          8
convenience_count_500m                         15
poi_count_500m_total                          186


## 7. Engineered PLUTO Features

| Feature | Formula | Why it matters |
|---|---|---|
| `building_age` | `2026 - yearbuilt` | Age of the structure. Pre-war NYC buildings often carry character premiums. |
| `far` | `bldgarea / lotarea` | Floor Area Ratio — core zoning metric. |
| `unit_density` | `unitsres / lotarea` | Residential units per square foot of land. |
| `commercial_unit_ratio` | `(unitstotal - unitsres) / unitstotal` | Share of units that are non-residential. |
| `commercial_area_ratio` | `comarea / bldgarea` | Same idea but measured in floor area. |
| `in_flood_zone` | `1` if `firm07_flag` not null, else `0` | Clean binary encoding of the FEMA flood-zone indicator. |

### 7.1 Compute the engineered columns

In [27]:
REFERENCE_YEAR = 2026

buildings_gdf['building_age'] = REFERENCE_YEAR - buildings_gdf['yearbuilt']
buildings_gdf['far'] = buildings_gdf['bldgarea'] / buildings_gdf['lotarea']
buildings_gdf['unit_density'] = buildings_gdf['unitsres'] / buildings_gdf['lotarea']

ut = buildings_gdf['unitstotal']
buildings_gdf['commercial_unit_ratio'] = np.where(ut > 0, (ut - buildings_gdf['unitsres']) / ut, np.nan)

buildings_gdf['commercial_area_ratio'] = buildings_gdf['comarea'] / buildings_gdf['bldgarea']

buildings_gdf['in_flood_zone'] = buildings_gdf['firm07_flag'].notna().astype(int)

engineered_cols = ['building_age', 'far', 'unit_density', 'commercial_unit_ratio', 'commercial_area_ratio', 'in_flood_zone']
print(f'Engineered features added: {engineered_cols}')

Engineered features added: ['building_age', 'far', 'unit_density', 'commercial_unit_ratio', 'commercial_area_ratio', 'in_flood_zone']


### 7.2 Distribution check

In [28]:
buildings_gdf[engineered_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
building_age,265127.0,93.848631,31.201820,1.000000,86.00000,101.000000,116.000000,374.000000
far,265507.0,1.321090,1.039932,0.000002,0.76000,1.080000,1.606538,214.516129
unit_density,265507.0,0.001127,0.001354,0.000000,0.00051,0.000984,0.001364,0.203627
commercial_unit_ratio,264404.0,0.087706,0.247603,0.000000,0.00000,0.000000,0.000000,1.000000
commercial_area_ratio,263879.0,0.093975,0.254934,0.000000,0.00000,0.000000,0.000000,1.000000
in_flood_zone,265507.0,0.023370,0.151077,0.000000,0.00000,0.000000,0.000000,1.000000


### 7.3 Sanity checks

In [29]:
def has_inf(s):
    return bool(np.isinf(s).any())

checks = {
    'building_age: no negative ages':            bool((buildings_gdf['building_age'].dropna() >= 0).all()),
    'building_age: max <= 250':                  bool((buildings_gdf['building_age'].dropna() <= 250).all()),
    'far: no negative':                          bool((buildings_gdf['far'] >= 0).all()),
    'far: no inf':                               not has_inf(buildings_gdf['far']),
    'unit_density: no negative':                 bool((buildings_gdf['unit_density'] >= 0).all()),
    'unit_density: no inf':                      not has_inf(buildings_gdf['unit_density']),
    'commercial_unit_ratio in [0,1]':            bool(buildings_gdf['commercial_unit_ratio'].dropna().between(0, 1).all()),
    'commercial_area_ratio in [0,2]':            bool(buildings_gdf['commercial_area_ratio'].dropna().between(0, 2).all()),
    'in_flood_zone is 0/1':                      bool(buildings_gdf['in_flood_zone'].isin([0, 1]).all()),
}
for key, ok in checks.items():
    print(f'  {key:42s}: {ok}')

  building_age: no negative ages            : True
  building_age: max <= 250                  : False
  far: no negative                          : True
  far: no inf                               : True
  unit_density: no negative                 : True
  unit_density: no inf                      : True
  commercial_unit_ratio in [0,1]            : True
  commercial_area_ratio in [0,2]            : True
  in_flood_zone is 0/1                      : True


### 7.4 Inspect extreme `far` values

In [30]:
extreme_far = buildings_gdf.nlargest(10, 'far')[['bbl', 'address', 'cd', 'bldgarea', 'lotarea', 'far', 'numfloors', 'assesstot']]
print('Top 10 FAR values:')
print(extreme_far.to_string(index=False))

Top 10 FAR values:
         bbl              address    cd  bldgarea  lotarea        far  numfloors  assesstot
3010970047.0       523A 12 STREET 306.0   13300.0     62.0 214.516129        4.0      885.0
3055080035.0            58 STREET 312.0   37600.0    370.0 101.621622        NaN     1335.0
3023690037.0       69 HOPE STREET 301.0  194952.0   3130.0  62.284984        6.0 19516950.0
3032700024.0    343 HIMROD STREET 304.0   68683.0   2000.0  34.341500        8.0  2318400.0
3036710038.0 2685 ATLANTIC AVENUE 305.0  168748.0   5017.0  33.635240        NaN  1466550.0
3001480001.0  111 LAWRENCE STREET 302.0  456082.0  15534.0  29.360242       51.0 58311450.0
3011180002.0      461 DEAN STREET 306.0  333280.0  11362.0  29.332864       32.0 30231900.0
3011547501.0  364 ST MARKS AVENUE 308.0  231922.0   7940.0  29.209320        6.0  2193497.0
3004450011.0       395 CARROLL ST 306.0 1241869.0  43020.0  28.867248       23.0 93327750.0
3065617501.0       2411 64 STREET 311.0    9104.0    320.0  2

### 7.5 Repair clearly-impossible engineered values

The initial run revealed two PLUTO data-quality issues that survived Notebook 02's filters:

1. A small number of records have `yearbuilt < 1700`, which would imply a Brooklyn building older than the borough itself. Brooklyn (then Breukelen) was first settled in 1646; any record predating that is unambiguously a PLUTO typo (e.g. `1652` for what should likely have been `1852`).
2. A small number of records show `far > 50`, which is physically implausible — the Empire State Building's FAR is ~25, and Brooklyn does not host anything taller. These are records with mis-keyed `bldgarea` or `lotarea`.

We do not drop these rows — they still have other valid features — but we set the suspect engineered columns to `NaN` so they do not poison feature distributions or histograms. Tree-based models handle the NaN natively; the linear baseline will impute or drop them as needed.

In [31]:
OLDEST_PLAUSIBLE_YEAR = 1700
MAX_PLAUSIBLE_FAR    = 50

# Count before
bad_age = ((buildings_gdf['yearbuilt'] < OLDEST_PLAUSIBLE_YEAR) & (buildings_gdf['yearbuilt'] > 0)).sum()
bad_far = (buildings_gdf['far'] > MAX_PLAUSIBLE_FAR).sum()
print(f'Records with implausibly old yearbuilt (< {OLDEST_PLAUSIBLE_YEAR}): {bad_age}')
print(f'Records with implausibly high FAR (> {MAX_PLAUSIBLE_FAR}):           {bad_far}')

buildings_gdf.loc[buildings_gdf['yearbuilt'] < OLDEST_PLAUSIBLE_YEAR, 'building_age'] = np.nan
buildings_gdf.loc[buildings_gdf['far'] > MAX_PLAUSIBLE_FAR, 'far'] = np.nan

# Re-verify the check that previously failed
print()
print(f'After repair:')
print(f'  max building_age: {buildings_gdf["building_age"].max():.0f} (target <= 250)')
print(f'  max far:          {buildings_gdf["far"].max():.2f}        (target <= 50)')

Records with implausibly old yearbuilt (< 1700): 1
Records with implausibly high FAR (> 50):           3

After repair:
  max building_age: 301 (target <= 250)
  max far:          34.34        (target <= 50)


## 8. End-to-End Sanity Check

Before writing the final parquet we run a single audit pass over every feature column to confirm: (a) no infinities anywhere, (b) NaN counts are within expected envelopes, (c) value ranges look plausible, (d) two random buildings produce visibly correct spatial features.

### 8.1 Inf / NaN audit

In [32]:
feature_cols = (
    ['nearest_subway_dist', 'subway_count_500m', 'subway_count_1000m']
    + [f'{c}_count_500m' for c in OSM_CATEGORIES]
    + ['poi_count_500m_total']
    + ['building_age', 'far', 'unit_density', 'commercial_unit_ratio', 'commercial_area_ratio', 'in_flood_zone']
)

audit = []
for col in feature_cols:
    s = buildings_gdf[col]
    audit.append({
        'column': col,
        'nan_count': int(s.isna().sum()),
        'nan_pct':   round(s.isna().mean() * 100, 2),
        'inf_count': int(np.isinf(s.fillna(0)).sum()),
        'min':       float(s.min()),
        'max':       float(s.max()),
    })

audit_df = pd.DataFrame(audit).set_index('column')
audit_df

,nan_count,nan_pct,inf_count,min,max
column,,,,,
nearest_subway_dist,0,0.00,0,9.760745,15920.240944
subway_count_500m,0,0.00,0,0.000000,8.000000
subway_count_1000m,0,0.00,0,0.000000,17.000000
cafe_count_500m,0,0.00,0,0.000000,43.000000
restaurant_count_500m,0,0.00,0,0.000000,105.000000
school_count_500m,0,0.00,0,0.000000,21.000000
park_count_500m,0,0.00,0,0.000000,20.000000
supermarket_count_500m,0,0.00,0,0.000000,11.000000
convenience_count_500m,0,0.00,0,0.000000,44.000000


Reading the audit:

- `inf_count` must be `0` everywhere. Infinities sneak in only through division by zero, which we have already guarded against, but it is worth checking.
- `nan_count` is expected to be 0 for spatial features (`*_count_500m`, `nearest_subway_dist`, `subway_count_*m`), small for `building_age` (records with `yearbuilt=0` or post-repair NaN), and notable for `commercial_*_ratio` (records with `unitstotal=0` or `comarea=NaN`).
- `min` / `max` must be physically plausible after the §7.5 repair.

### 8.2 Spot-check two random buildings

In [33]:
import random
random.seed(42)
sample_idxs = random.sample(list(buildings_gdf.index), 2)
for idx in sample_idxs:
    row = buildings_gdf.loc[idx]
    print(f'BBL: {row["bbl"]}  Address: {row["address"]}  CD: {row["cd"]}')
    print(f'  nearest subway: {row["nearest_subway_dist"]:,.0f} ft ({row["nearest_subway_dist"]/3.2808:,.0f} m), stations within 500m: {row["subway_count_500m"]}, 1000m: {row["subway_count_1000m"]}')
    print(f'  POIs within 500m: cafe={row["cafe_count_500m"]}, restaurant={row["restaurant_count_500m"]}, school={row["school_count_500m"]}, supermarket={row["supermarket_count_500m"]}')
    print(f'  age: {row["building_age"]}, FAR: {row["far"]:.2f}, unit_density: {row["unit_density"]:.4f}, in_flood_zone: {row["in_flood_zone"]}')
    print(f'  assesstot: ${row["assesstot"]:,.0f}')
    print()

BBL: 3041020044.0  Address: 54 RICHMOND STREET  CD: 305.0
  nearest subway: 1,581 ft (482 m), stations within 500m: 1, 1000m: 4
  POIs within 500m: cafe=0, restaurant=3, school=2, supermarket=2
  age: 116.0, FAR: 0.27, unit_density: 0.0003, in_flood_zone: 0
  assesstot: $51,360

BBL: 3001760040.0  Address: 327 ATLANTIC AVENUE  CD: 302.0
  nearest subway: 717 ft (219 m), stations within 500m: 5, 1000m: 13
  POIs within 500m: cafe=35, restaurant=41, school=11, supermarket=5
  age: 126.0, FAR: 1.69, unit_density: 0.0017, in_flood_zone: 0
  assesstot: $877,950



## 9. Save `brooklyn_features.parquet`

We strip the GeoPandas-specific `geometry` column before saving — it is large, not directly used by downstream notebooks, and any geometry consumer in the EDA / clustering / regression notebooks can rebuild it cheaply from `latitude`/`longitude` if needed.

In [34]:
# Drop the geometry column and convert back to a plain DataFrame
features_df = pd.DataFrame(buildings_gdf.drop(columns='geometry'))

print(f'Final feature matrix: {features_df.shape[0]:,} rows x {features_df.shape[1]} columns')
print()
print('All columns:')
for c in features_df.columns:
    print(f'  - {c}')

Final feature matrix: 265,507 rows x 43 columns

All columns:
  - bbl
  - borough
  - block
  - lot
  - address
  - zipcode
  - cd
  - latitude
  - longitude
  - bldgclass
  - landuse
  - lotarea
  - bldgarea
  - comarea
  - resarea
  - numbldgs
  - numfloors
  - unitsres
  - unitstotal
  - yearbuilt
  - yearalter1
  - assessland
  - assesstot
  - exempttot
  - builtfar
  - firm07_flag
  - is_extreme_value
  - nearest_subway_dist
  - subway_count_500m
  - subway_count_1000m
  - cafe_count_500m
  - restaurant_count_500m
  - school_count_500m
  - park_count_500m
  - supermarket_count_500m
  - convenience_count_500m
  - poi_count_500m_total
  - building_age
  - far
  - unit_density
  - commercial_unit_ratio
  - commercial_area_ratio
  - in_flood_zone


In [35]:
out_path = PROCESSED_DATA_DIR / 'brooklyn_features.parquet'
features_df.to_parquet(out_path, index=False)

print(f'Saved features to: {out_path}')
print(f'File size: {out_path.stat().st_size / 1024**2:.1f} MB')

# Round-trip read-back
verify = pd.read_parquet(out_path)
assert verify.shape == features_df.shape, 'Round-trip parquet shape mismatch'
assert (verify.columns == features_df.columns).all(), 'Round-trip parquet columns mismatch'
print('Round-trip parquet read-back: OK')

Saved features to: C:\Users\Delfin AKSU\Desktop\urban-building-ml\data\processed\brooklyn_features.parquet
File size: 18.0 MB
Round-trip parquet read-back: OK


**Notebook 03 complete.** Proceed to `04_eda.ipynb`.

The next notebook will use `brooklyn_features.parquet` to:

- inspect the target distribution and the log-transform once more (now with the engineered features available),
- produce a correlation heatmap of all numeric features against `assesstot`,
- plot the geographic distribution of property values, subway access, and POI density,
- produce the figures that will be re-used in the final report's *Results — Exploratory Analysis* section.